# 05. Recommendation Engine

## Objective

While the machine learning model predicts the priority of a traffic event, operational teams still need guidance on the actions required after a prediction.

This notebook develops a rule-based recommendation engine that converts model outputs into actionable traffic management recommendations.

---

## Recommendation Pipeline

Traffic Event
        ↓
Priority Prediction
        ↓
Impact Score
        ↓
Impact Level
        ↓
Operational Recommendations

---

## Business Logic

An **Impact Score** is computed using domain-specific rules based on:

- Event Priority
- Peak Hour
- Road Closure Requirement
- Event Type (Planned / Unplanned)
- Event Category
- Weekend Adjustment

The score is capped between **0 and 100**.

---

## Impact Level Classification

| Impact Score | Impact Level |
|--------------|--------------|
| 0–30 | Low |
| 31–60 | Medium |
| 61–100 | High |

---

## Operational Recommendations

Based on the Impact Level, the system recommends:

- Number of Traffic Officers
- Number of Barricades
- Traffic Diversion Requirement
- Expected Response Time
- Traffic Advisory

---

## Recommendation Rules

| Impact Level | Officers | Barricades | Diversion | Response Time |
|--------------|----------|------------|-----------|---------------|
| Low | 2 | 2 | No | Within 30 minutes |
| Medium | 4 | 4 | No | Within 20 minutes |
| High | 6 | 8 | Yes | Immediate (5–10 minutes) |

---

## Generated Outputs

For every traffic event, the recommendation engine generates:

- Impact Score
- Impact Level
- Officers Required
- Barricades Required
- Diversion Recommendation
- Response Time
- Traffic Advisory
- Human-readable Recommendation Summary

---

## Outcome

The recommendation engine transforms machine learning predictions into actionable operational decisions, enabling traffic authorities to allocate resources more efficiently and respond consistently to different traffic events.

This demonstrates how predictive analytics can be combined with rule-based decision support to build an intelligent traffic management system.

###Load Feature Engineered Data

In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv("/content/feature_engineered_events.csv")

print(df.shape)
df.head()

(8171, 23)


,event_type,latitude,longitude,address,event_cause,requires_road_closure,start_datetime,authenticated,modified_datetime,description,...,created_date,police_station,zone,junction,hour,day_of_week,month,is_weekend,is_peak_hour,event_category
0,unplanned,13.040004,77.518099,"Mumbai Bengaluru Highway, Jalahalli Cross Junc...",vehicle_breakdown,False,2024-03-07 17:01:48.111000+00:00,yes,2024-03-07 19:35:47.871698+00:00,s m circle in coming man track,...,2024-03-07 17:03:51.164032+00:00,Peenya,Unknown,Unknown,17,Thursday,March,0,1,Incident
1,unplanned,12.921876,77.645158,"19th Main Road, Heavie Halcyon, Agara, HSR Lay...",vehicle_breakdown,False,2024-01-30 04:07:24.173000+00:00,yes,2024-01-30 04:17:46.828979+00:00,Starting problem,...,2024-01-30 04:08:22.954979+00:00,HSR Layout,Unknown,Unknown,4,Tuesday,January,0,0,Incident
2,unplanned,12.955622,77.585708,"Lalbagh Main Road, Dr Sri Shantaveera Swami Ci...",others,False,2023-11-11 06:18:03.343000+00:00,yes,2024-01-30 04:56:03.282003+00:00,ಊರ್ವಶಿ ಜಂಕ್ಷನ್ ನಲ್ಲಿ ಒಳಚರಂಡಿ ಚೇಂಬರ್ ಗೆ ಹೊಸದಾಗಿ...,...,2023-11-11 06:20:00.989398+00:00,Wilson Garden,Central Zone 2,UrvashiJunction,6,Saturday,November,1,0,Other
3,unplanned,13.006147,77.579435,"Sankey Road, Bashyam Circle, Sadashiva Nagar, ...",tree_fall,True,2024-03-07 17:56:55.061000+00:00,yes,2024-03-14 07:42:05.550050+00:00,tree fall,...,2024-03-07 17:58:56.696892+00:00,Sadashivanagar,Unknown,Unknown,17,Thursday,March,0,1,Natural
4,unplanned,12.953980,77.585233,"Lalbagh Fort Road, Lalbagh Main Gate Junction,...",vehicle_breakdown,False,2024-01-30 04:56:32.348000+00:00,yes,2024-01-30 05:35:17.339080+00:00,[LOCATION] ಪೈಪ್ [PERSON] ವಾಹನ ಆಫ್ ಆಗಿರುತ್ತದೆ ಸರ್,...,2024-01-30 04:58:55.937662+00:00,Wilson Garden,Unknown,LalbaghMainGateJunc,4,Tuesday,January,0,0,Incident


In [20]:
def calculate_impact_score(row):
    score = 0

    # Priority
    priority_score = {
        "High": 40,
        "Low": 0
    }
    score += priority_score.get(row["priority"], 0)

    # Road Closure
    if row["requires_road_closure"]:
        score += 25

    # Peak Hour
    if row["is_peak_hour"]:
        score += 20

    # Event Type
    event_type_score = {
        "unplanned": 10,
        "planned": 0
    }
    score += event_type_score.get(row["event_type"], 0)

    # Event Category
    if row["event_category"] in ["Infrastructure", "Natural"]:
        score += 5

    # Weekend Adjustment
    if row["is_weekend"]:
        score -= 5

    return min(score, 100)

In [21]:
df["impact_score"] = df.apply(calculate_impact_score, axis=1)

In [22]:
def get_impact_level(score):
    if score <= 30:
        return "Low"
    elif score <= 60:
        return "Medium"
    else:
        return "High"

In [23]:
df["impact_level"] = df["impact_score"].apply(get_impact_level)

In [25]:
df["impact_level"].value_counts()

,count
impact_level,
Medium,3169
Low,2700
High,2302


In [26]:
officer_mapping = {
    "Low": 2,
    "Medium": 4,
    "High": 6
}

df["officers_required"] = df["impact_level"].map(officer_mapping)

In [27]:
df[["impact_level", "officers_required"]].drop_duplicates()

,impact_level,officers_required
0,High,6
1,Medium,4
2,Low,2


In [28]:
barricade_mapping = {
    "Low": 2,
    "Medium": 4,
    "High": 8
}

df["barricades_required"] = df["impact_level"].map(barricade_mapping)

In [29]:
df[["impact_level", "barricades_required"]].drop_duplicates()

,impact_level,barricades_required
0,High,8
1,Medium,4
2,Low,2


In [30]:
def diversion_required(row):

    if row["impact_level"] == "High":
        return True

    return False

In [31]:
df["diversion_required"] = df.apply(diversion_required, axis=1)

In [32]:
df[
    [
        "impact_level",
        "officers_required",
        "barricades_required",
        "diversion_required"
    ]
].head(10)

,impact_level,officers_required,barricades_required,diversion_required
0,High,6,8,True
1,Medium,4,4,False
2,Low,2,2,False
3,Medium,4,4,False
4,Low,2,2,False
5,Low,2,2,False
6,Low,2,2,False
7,Low,2,2,False
8,Medium,4,4,False
9,Low,2,2,False


In [33]:
response_time_mapping = {
    "Low": "Within 30 minutes",
    "Medium": "Within 20 minutes",
    "High": "Immediate (5–10 minutes)"
}

df["response_time"] = df["impact_level"].map(response_time_mapping)

In [34]:
df[["impact_level", "response_time"]].drop_duplicates()

,impact_level,response_time
0,High,Immediate (5–10 minutes)
1,Medium,Within 20 minutes
2,Low,Within 30 minutes


In [35]:
def generate_advisory(level):

    if level == "High":
        return "Heavy traffic expected. Deploy emergency response and activate diversions."

    elif level == "Medium":
        return "Moderate congestion expected. Monitor traffic and dispatch additional officers."

    else:
        return "Minor traffic disruption. Continue routine monitoring."

In [36]:
df["traffic_advisory"] = df["impact_level"].apply(generate_advisory)

In [37]:
def generate_recommendation(row):

    recommendation = (
        f"Deploy {row['officers_required']} traffic officers, "
        f"place {row['barricades_required']} barricades, "
        f"{'activate traffic diversion' if row['diversion_required'] else 'no diversion required'}, "
        f"response time: {row['response_time']}."
    )

    return recommendation

In [38]:
df["recommendation"] = df.apply(generate_recommendation, axis=1)

In [40]:
df[
    [
        "priority",
        "impact_score",
        "impact_level",
        "officers_required",
        "barricades_required",
        "diversion_required",
        "response_time",
        "traffic_advisory",
        "recommendation"
    ]
].head(10)

,priority,impact_score,impact_level,officers_required,barricades_required,diversion_required,response_time,traffic_advisory,recommendation
0,High,70,High,6,8,True,Immediate (5–10 minutes),Heavy traffic expected. Deploy emergency respo...,"Deploy 6 traffic officers, place 8 barricades,..."
1,High,50,Medium,4,4,False,Within 20 minutes,Moderate congestion expected. Monitor traffic ...,"Deploy 4 traffic officers, place 4 barricades,..."
2,Low,5,Low,2,2,False,Within 30 minutes,Minor traffic disruption. Continue routine mon...,"Deploy 2 traffic officers, place 2 barricades,..."
3,Low,60,Medium,4,4,False,Within 20 minutes,Moderate congestion expected. Monitor traffic ...,"Deploy 4 traffic officers, place 4 barricades,..."
4,Low,10,Low,2,2,False,Within 30 minutes,Minor traffic disruption. Continue routine mon...,"Deploy 2 traffic officers, place 2 barricades,..."
5,Low,10,Low,2,2,False,Within 30 minutes,Minor traffic disruption. Continue routine mon...,"Deploy 2 traffic officers, place 2 barricades,..."
6,Low,30,Low,2,2,False,Within 30 minutes,Minor traffic disruption. Continue routine mon...,"Deploy 2 traffic officers, place 2 barricades,..."
7,Low,10,Low,2,2,False,Within 30 minutes,Minor traffic disruption. Continue routine mon...,"Deploy 2 traffic officers, place 2 barricades,..."
8,High,40,Medium,4,4,False,Within 20 minutes,Moderate congestion expected. Monitor traffic ...,"Deploy 4 traffic officers, place 4 barricades,..."
9,Low,10,Low,2,2,False,Within 30 minutes,Minor traffic disruption. Continue routine mon...,"Deploy 2 traffic officers, place 2 barricades,..."
